In [ ]:
import subprocess, os, matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

def run_tshark_count(pcap_path: Path, display_filter: str) -> int:
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-Y", display_filter,
        "-T", "fields", "-e", "frame.number"
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True, check=True)
    lines = result.stdout.strip().splitlines()
    return len(lines) if lines and lines[0] else 0

cellreplay_dirs = [
    # Path for CSV files
]

mahimahi_dirs = [
    # Path for CSV files
]

mahimahi_4g_dirs = [
    # Path for CSV files
]

client_ip = "10.0.16.2"

records = []
for category, folder_list in [
        ("CellReplay",   cellreplay_dirs),
        ("Mahimahi",     mahimahi_dirs),
        ("4G_Mahimahi",  mahimahi_4g_dirs)]:

    for folder in folder_list:
        if not os.path.exists(folder):
            print(f"Folder does not exist: {folder}")
            continue

        for file in os.listdir(folder):
            if file.lower().endswith((".pcap", ".pcapng")):
                pcap_path = Path(folder) / file
                print(f"Analyzing: {pcap_path.name}")

                try:
                    record = {
                        "Category"          : category,
                        "Algorithm"         : Path(file).stem,
                        "File"              : file,
                        "Path"              : folder,
                        "Retransmissions"   : run_tshark_count(pcap_path,
                                                               "tcp.analysis.retransmission"),
                        "Total TCP Packets" : run_tshark_count(pcap_path, "tcp"),
                        "TCP Data (bytes>0)": run_tshark_count(pcap_path, "tcp.len > 0"),
                        "Sent Packets"      : run_tshark_count(
                                                pcap_path,
                                                f"ip.src == {client_ip} and tcp.len > 0"),
                        "Received Packets"  : run_tshark_count(
                                                pcap_path,
                                                f"ip.dst == {client_ip} and tcp.len > 0"),
                    }
                    records.append(record)
                except subprocess.CalledProcessError as e:
                    print(f"tshark error on {pcap_path}:\n{e.stderr}")

metrics_df = pd.DataFrame(records)

print("Combined Results")
display(metrics_df)

cellreplay_df = metrics_df[metrics_df["Category"] == "CellReplay"  ].reset_index(drop=True)
mahimahi_df   = metrics_df[metrics_df["Category"] == "Mahimahi"    ].reset_index(drop=True)
fourg_df      = metrics_df[metrics_df["Category"] == "4G_Mahimahi"].reset_index(drop=True)

print("CellReplay Results");  display(cellreplay_df)
print("Mahimahi Results");    display(mahimahi_df)
print("4G Mahimahi Results"); display(fourg_df)


In [ ]:
metrics_df["Retrans %"] = (
    metrics_df["Retransmissions"] / metrics_df["Total TCP Packets"] * 100
).round(2)

def pct_table(category):
    return metrics_df[metrics_df["Category"] == category][
        ["Algorithm", "Retransmissions", "Total TCP Packets", "Retrans %"]
    ].reset_index(drop=True)

print("CellReplay – % Retrans");  display(pct_table("CellReplay"))
print("Mahimahi – % Retrans ===");    display(pct_table("Mahimahi"))
print("4G Mahimahi – % Retrans"); display(pct_table("4G_Mahimahi"))


In [ ]:
import numpy as np

def get_retrans_times(pcap_path: Path):
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-Y", "tcp.analysis.retransmission",
        "-T", "fields", "-e", "frame.time_epoch"
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            text=True, check=True)
    return [float(ts) for ts in result.stdout.strip().splitlines() if ts]

gap_records = []
for row in metrics_df.itertuples(index=False):
    pcap_path = Path(row.Path) / row.File
    try:
        ts_list = get_retrans_times(pcap_path)
        if len(ts_list) >= 2:
            first_gaps = round(ts_list[1] - ts_list[0], 2)
            consecutive = [round(j-i, 2) for i, j in zip(ts_list[:-1], ts_list[1:])]
        else:
            first_gaps = None
            consecutive = []

        gap_records.append({
            "Category"           : row.Category,
            "Algorithm"          : row.Algorithm,
            "File"               : row.File,
            "Total Retrans"      : len(ts_list),
            "First Gap (s)"      : first_gaps,
            "Δ Consecutive (s)"  : consecutive,
        })
    except subprocess.CalledProcessError:
        pass

retrans_gap_df = pd.DataFrame(gap_records)
print("Retransmission Intervals (all)")
display(retrans_gap_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import subprocess

def get_first_gap(pcap_path: Path):
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-Y", "tcp.analysis.retransmission",
        "-T", "fields", "-e", "frame.time_epoch"
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True, check=True)
    ts = [float(x) for x in result.stdout.strip().splitlines() if x]
    return round(ts[1] - ts[0], 2) if len(ts) >= 2 else None

gap_map = {}
for row in metrics_df.itertuples(index=False):
    pcap_path = Path(row.Path) / row.File
    try:
        gap = get_first_gap(pcap_path)
        if gap is not None:
            gap_map.setdefault(row.Category, []).append(gap)
    except subprocess.CalledProcessError:
        continue

pdf_dir = Path(r"") # Path for pdf directory
pdf_dir.mkdir(parents=True, exist_ok=True)

for cat, gaps in gap_map.items():
    if not gaps:
        continue

    fig, ax = plt.subplots()
    fig.set_size_inches(5, 3)

    ax.hist(gaps, bins=10, edgecolor='black')

    ax.set_xlabel("Gap (s)")
    ax.set_ylabel("Frequency")
    ax.grid(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(pdf_dir / f"retrans_gap_{cat}.pdf", format="pdf", bbox_inches="tight")
    plt.show()

